# ⚾ KIA는 몇 도에서 가장 강할까?
### 2024~2026년 KIA 정규시즌 432경기 — 그래프 만드는 방법

**사용법**: 이 노트북과 함께 제공한 CSV를 같은 폴더에 넣고 `Shift + Enter`로 위에서부터 실행하세요.

**만드는 그래프**
1. 기온별 승률 막대그래프
2. 연도별 기온·승률 비교그래프
3. 기온별 평균 득점 그래프

**승률 기준:** `승 / (승 + 패) × 100` (무승부는 분모에서 제외). 기상관측소 기온은 경기장 현장 기온과 다를 수 있습니다.

## 1단계. 필요한 도구 불러오기
`pandas`는 표 계산, `matplotlib`은 그래프를 그리는 도구입니다. 한글이 깨지지 않도록 글꼴을 설정합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 윈도우에서는 맑은 고딕, 다른 운영체제에서는 설치된 한글 글꼴 사용
import platform
plt.rcParams['font.family'] = 'Malgun Gothic' if platform.system() == 'Windows' else 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

# 완성된 그래프를 저장할 폴더
Path('그래프_결과').mkdir(exist_ok=True)
print('준비 완료!')

## 2단계. CSV 불러오기
`pd.read_csv()`는 CSV를 표로 읽습니다. 파일이 없으면 두 파일이 같은 폴더에 있는지 확인하세요.

In [ ]:
df = pd.read_csv('KIA_2024_2026_432경기_기온분석_원본.csv', encoding='utf-8-sig')
print('전체 경기 수:', len(df))
print('연도별 경기 수:')
print(df.groupby('연도').size())
display(df.head(3))

## 3단계. 기온 구간 만들기
`pd.cut()`으로 기온을 6개 구간으로 나눕니다. `right=False`는 **왼쪽 경계 포함, 오른쪽 경계 제외**를 뜻합니다. 예: 25.0℃는 `25~30℃`.

In [ ]:
bins = [-float('inf'), 10, 15, 20, 25, 30, float('inf')]
labels = ['10℃ 미만', '10~15℃', '15~20℃', '20~25℃', '25~30℃', '30℃ 이상']
df['기온구간'] = pd.cut(df['기온(°C)'], bins=bins, labels=labels, right=False)
print(df['기온구간'].value_counts(sort=False))

## 4단계. 승률과 평균 득점 계산하기
`groupby()`는 같은 기온 구간의 경기를 모읍니다. `승률`은 무승부를 빼고 계산하고, `평균득점`은 해당 구간 전체 경기의 평균입니다.

In [ ]:
def summarize(data):
    summary = data.groupby('기온구간', observed=False).agg(
        경기수=('경기ID', 'count'),
        승=('경기결과', lambda x: (x == '승').sum()),
        패=('경기결과', lambda x: (x == '패').sum()),
        무=('경기결과', lambda x: (x == '무').sum()),
        평균득점=('KIA득점', 'mean')
    )
    summary['승률(%)'] = (summary['승'] / (summary['승'] + summary['패']) * 100).round(2)
    summary['평균득점'] = summary['평균득점'].round(2)
    return summary

result = summarize(df)
display(result)

## 5단계. 그래프 ① 기온별 승률
**막대의 높이**는 승률(%), **막대 위 숫자**는 승률, **아래 괄호**는 경기 수입니다. `plt.bar()`로 막대그래프를 만듭니다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(labels, result['승률(%)'], color='#C83835', width=0.65)
ax.set_title('KIA 2024~2026 기온별 승률', fontsize=17, fontweight='bold')
ax.set_ylabel('승률 (%)')
ax.set_ylim(0, 78)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
for bar, rate, count in zip(bars, result['승률(%)'], result['경기수']):
    if pd.notna(rate):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{rate:.1f}%\n({count}경기)', ha='center', fontsize=10)
fig.tight_layout()
fig.savefig('그래프_결과/01_기온별승률.png', dpi=180, bbox_inches='tight')
plt.show()

**읽는 방법:** 막대가 높을수록 해당 기온 구간의 승률이 높습니다. 단, 10℃ 미만은 3경기뿐이므로 일반화하면 안 됩니다.

## 6단계. 그래프 ② 연도별 기온·승률 비교
각 연도마다 같은 방식으로 승률을 계산하고, **나란히 있는 막대**로 비교합니다. 3년의 승률 차이를 한눈에 볼 수 있습니다.

In [ ]:
yearly = []
for year in [2024, 2025, 2026]:
    one = summarize(df[df['연도'] == year])
    yearly.append(one['승률(%)'].rename(str(year)))
compare = pd.concat(yearly, axis=1)
display(compare)

fig, ax = plt.subplots(figsize=(12, 5.5))
compare.plot(kind='bar', ax=ax, color=['#A4A9B3', '#C83835', '#E5A35B'], width=0.78)
ax.set_title('KIA 연도별 기온 구간 승률 비교', fontsize=16, fontweight='bold')
ax.set_ylabel('승률 (%)')
ax.set_xlabel('기온 구간')
ax.set_xticklabels(labels, rotation=0)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
ax.legend(title='시즌')
fig.tight_layout()
fig.savefig('그래프_결과/02_연도별기온승률.png', dpi=180, bbox_inches='tight')
plt.show()

**읽는 방법:** 같은 기온에서도 시즌마다 승률이 다를 수 있습니다. 표본이 적은 구간은 특히 변동이 큽니다.

## 7단계. 그래프 ③ 기온별 평균 득점
승률 대신 **KIA 평균 득점**을 비교합니다. `plt.bar()`는 위와 같고, y축 값만 평균 득점으로 바뀝니다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(labels, result['평균득점'], color='#276A8C', width=0.65)
ax.set_title('KIA 2024~2026 기온별 평균 득점', fontsize=17, fontweight='bold')
ax.set_ylabel('경기당 평균 득점 (점)')
ax.set_ylim(0, max(result['평균득점'].fillna(0)) + 1.7)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
for bar, score in zip(bars, result['평균득점']):
    if pd.notna(score):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.12,
                f'{score:.2f}점', ha='center', fontsize=10)
fig.tight_layout()
fig.savefig('그래프_결과/03_기온별평균득점.png', dpi=180, bbox_inches='tight')
plt.show()

## 8단계. 그래프에 사용한 계산표 저장하기
엑셀에서 열 수 있는 CSV로 저장합니다. `utf-8-sig`는 윈도우 엑셀에서 한글 깨짐을 줄여줍니다.

In [ ]:
result.to_csv('그래프_결과/기온별_승률_득점_계산표.csv', encoding='utf-8-sig')
compare.to_csv('그래프_결과/연도별_기온승률_계산표.csv', encoding='utf-8-sig')
print('완료! 그래프_결과 폴더에 PNG 3개와 CSV 2개가 저장됐습니다.')

## 발표용 설명과 주의사항

- **분석 질문:** KIA는 어느 기온 구간에서 승률이 가장 높았는가?
- **방법:** 2024~2026 정규시즌 432경기의 관측소 기온을 6개 구간으로 분류하고, 구간별 승률과 평균 득점을 계산했습니다.
- **핵심 해석:** 승률이 가장 높은 구간을 찾되, 그 차이가 기온 때문에 생겼다고 단정할 수 없습니다.
- **주의:** 홈/원정, 상대팀, 선수 구성, 경기장 등 다른 요인도 승패에 영향을 줍니다. 강수량 공란은 0mm로 대체하지 않았습니다.
- **검증:** 이 노트북은 제공된 통합 CSV의 숫자를 계산합니다. KBO·기상청 원본을 새로 내려받는 코드는 아닙니다.